# Flu Shot Learning: Cross-Validated Ensemble

This experiment improves the previous CatBoost submission with leakage-free survey feature engineering, five-fold validation, target-specific models, and a small ensemble of CatBoost configurations. The metric is the mean ROC AUC for the H1N1 and seasonal vaccine targets.

In [1]:
from pathlib import Path
import importlib.util
import sys

if importlib.util.find_spec('catboost') is None:
    !{sys.executable} -m pip install -q catboost

import numpy as np
import pandas as pd
from catboost import CatBoostClassifier
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import StratifiedKFold

RANDOM_STATE = 2026
N_SPLITS = 5
TARGETS = ['h1n1_vaccine', 'seasonal_vaccine']
pd.set_option('display.max_columns', 100)

## 1. Load competition data

In [2]:
def find_data_dir():
    candidates = [
        Path('/kaggle/input/datasets/barshamishra19/flu-shot-learning-dataset'),
        Path('/kaggle/input/datasets/barshamishra19/flu-shot-learning-dataset'),
        Path('/kaggle/input/datasets/barshamishra19/flu-shot-learning-dataset'),
        Path.cwd() / 'data' / 'raw',
        Path.cwd().parent / 'data' / 'raw',
        Path.cwd().parent.parent.parent / 'data' / 'raw',
    ]
    for candidate in candidates:
        if (candidate / 'training_set_features.csv').exists():
            return candidate
    raise FileNotFoundError('Set DATA_DIR to the folder containing the competition CSV files.')

DATA_DIR = find_data_dir()
train_features = pd.read_csv(DATA_DIR / 'training_set_features.csv')
train_labels = pd.read_csv(DATA_DIR / 'training_set_labels.csv')
test_features = pd.read_csv(DATA_DIR / 'test_set_features.csv')
assert train_features['respondent_id'].equals(train_labels['respondent_id'])
print(f'Using: {DATA_DIR.resolve()}')
print(f'Train: {train_features.shape}; test: {test_features.shape}')
display(train_labels[TARGETS].mean().rename('positive_rate').to_frame())

Using: /kaggle/input/datasets/barshamishra19/flu-shot-learning-dataset
Train: (26707, 36); test: (26708, 36)


,positive_rate
h1n1_vaccine,0.212454
seasonal_vaccine,0.465608


## 2. Engineer leakage-free survey features

The added features summarize behavior, opinions, missingness, and ordered survey responses. No target values are used.

In [3]:
def engineer_features(frame):
    result = frame.drop(columns=['respondent_id'], errors='ignore').copy()
    result['missing_count'] = result.isna().sum(axis=1)
    result['missing_fraction'] = result.isna().mean(axis=1)

    behavior_columns = [c for c in result.columns if c.startswith('behavioral_')]
    result['behavior_score'] = result[behavior_columns].sum(axis=1, min_count=1)
    result['behavior_count_available'] = result[behavior_columns].notna().sum(axis=1)

    opinion_columns = [c for c in result.columns if c.startswith('opinion_')]
    result['opinion_score'] = result[opinion_columns].mean(axis=1)
    result['opinion_risk_score'] = result[[c for c in opinion_columns if '_risk' in c]].mean(axis=1)
    result['doctor_recommendation_count'] = result[['doctor_recc_h1n1', 'doctor_recc_seasonal']].sum(axis=1, min_count=1)
    result['health_access_score'] = result[['health_insurance', 'health_worker']].sum(axis=1, min_count=1)

    ordinal_maps = {
        'age_group': {'18 - 34 Years': 1, '35 - 44 Years': 2, '45 - 54 Years': 3, '55 - 64 Years': 4, '65+ Years': 5},
        'education': {'< 12 Years': 1, '12 Years': 2, 'Some College': 3, 'College Graduate': 4},
        'income_poverty': {'Below Poverty': 1, '<= $75,000, Above Poverty': 2, '> $75,000': 3},
    }
    for column, mapping in ordinal_maps.items():
        if column in result:
            result[column + '_ordinal'] = result[column].map(mapping)
    return result

X = engineer_features(train_features)
X_test = engineer_features(test_features)
categorical_columns = X.select_dtypes(include=['object']).columns.tolist()
for column in categorical_columns:
    X[column] = X[column].fillna('__MISSING__').astype(str)
    X_test[column] = X_test[column].fillna('__MISSING__').astype(str)
cat_feature_indices = [X.columns.get_loc(column) for column in categorical_columns]
y = train_labels[TARGETS].copy()
stratify_key = y[TARGETS].astype(str).agg('_'.join, axis=1)
print(f'Features: {X.shape[1]}; categorical: {len(categorical_columns)}')

Features: 46; categorical: 12


## 3. Compare CatBoost configurations with five-fold out-of-fold validation

In [4]:
model_configs = {
    'balanced': {'depth': 7, 'learning_rate': 0.04, 'l2_leaf_reg': 6.0},
    'regularized': {'depth': 6, 'learning_rate': 0.055, 'l2_leaf_reg': 10.0},
}
target_params = {
    'h1n1_vaccine': {'iterations': 1800},
    'seasonal_vaccine': {'iterations': 1600},
}

def make_model(config_name, target, seed):
    params = model_configs[config_name] | target_params[target]
    return CatBoostClassifier(
        **params, loss_function='Logloss', eval_metric='AUC', random_seed=seed,
        verbose=False, allow_writing_files=False, thread_count=-1,
    )

folds = list(StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_STATE).split(X, stratify_key))
oof = {name: pd.DataFrame(index=X.index, columns=TARGETS, dtype=float) for name in model_configs}

for config_name in model_configs:
    for fold_number, (train_index, valid_index) in enumerate(folds, start=1):
        for target in TARGETS:
            model = make_model(config_name, target, RANDOM_STATE + fold_number)
            model.fit(
                X.iloc[train_index], y.iloc[train_index][target],
                cat_features=cat_feature_indices,
                eval_set=(X.iloc[valid_index], y.iloc[valid_index][target]),
                use_best_model=True, early_stopping_rounds=120,
            )
            oof[config_name].loc[valid_index, target] = model.predict_proba(X.iloc[valid_index])[:, 1]
    scores = [roc_auc_score(y[target], oof[config_name][target]) for target in TARGETS]
    print(config_name, dict(zip(TARGETS, np.round(scores, 5))), 'mean:', round(float(np.mean(scores)), 5))

balanced {'h1n1_vaccine': np.float64(0.87129), 'seasonal_vaccine': np.float64(0.86477)} mean: 0.86803
regularized {'h1n1_vaccine': np.float64(0.87148), 'seasonal_vaccine': np.float64(0.86492)} mean: 0.8682


## 4. Select a conservative OOF blend

In [5]:
blend_results = []
for weight in np.linspace(0.0, 1.0, 11):
    blend_scores = []
    for target in TARGETS:
        prediction = weight * oof['balanced'][target] + (1.0 - weight) * oof['regularized'][target]
        blend_scores.append(roc_auc_score(y[target], prediction))
    blend_results.append((weight, float(np.mean(blend_scores)), blend_scores))
best_weight, best_score, best_target_scores = max(blend_results, key=lambda row: row[1])
print(f'Best balanced weight: {best_weight:.2f}')
print(dict(zip(TARGETS, np.round(best_target_scores, 5))))
print(f'OOF mean ROC AUC: {best_score:.5f}')

Best balanced weight: 0.50
{'h1n1_vaccine': np.float64(0.87182), 'seasonal_vaccine': np.float64(0.86518)}
OOF mean ROC AUC: 0.86850


## 5. Train fold ensemble and create submission

Each final prediction is the average of five fold models and two random seeds. This uses all rows for training while preserving the variance reduction from the fold ensemble.

In [6]:
test_predictions = {target: np.zeros(len(X_test)) for target in TARGETS}
for config_name, config_weight in [('balanced', best_weight), ('regularized', 1.0 - best_weight)]:
    if config_weight == 0:
        continue
    for fold_number, (train_index, valid_index) in enumerate(folds, start=1):
        for target in TARGETS:
            fold_predictions = []
            for seed_offset in (0, 101):
                model = make_model(config_name, target, RANDOM_STATE + fold_number + seed_offset)
                model.fit(X.iloc[train_index], y.iloc[train_index][target], cat_features=cat_feature_indices, verbose=False)
                fold_predictions.append(model.predict_proba(X_test)[:, 1])
            test_predictions[target] += config_weight * np.mean(fold_predictions, axis=0) / N_SPLITS

submission = pd.DataFrame({
    'respondent_id': test_features['respondent_id'],
    'h1n1_vaccine': np.clip(test_predictions['h1n1_vaccine'], 0.0, 1.0),
    'seasonal_vaccine': np.clip(test_predictions['seasonal_vaccine'], 0.0, 1.0),
})
assert list(submission.columns) == ['respondent_id', 'h1n1_vaccine', 'seasonal_vaccine']
assert submission['respondent_id'].equals(test_features['respondent_id'])
assert submission[TARGETS].notna().all().all()
assert ((submission[TARGETS] >= 0) & (submission[TARGETS] <= 1)).all().all()
output_path = Path('submission_ensemble.csv')
submission.to_csv(output_path, index=False)
print(f'Saved {len(submission):,} rows to {output_path.resolve()}')
display(submission.head())
display(submission[TARGETS].describe().loc[['min', 'mean', 'max']])

Saved 26,708 rows to /kaggle/working/submission_ensemble.csv


,respondent_id,h1n1_vaccine,seasonal_vaccine
0,26707,0.103105,0.152785
1,26708,0.025681,0.027597
2,26709,0.197794,0.759284
3,26710,0.689383,0.901806
4,26711,0.325489,0.441673


,h1n1_vaccine,seasonal_vaccine
min,0.001022,0.003513
mean,0.206699,0.465219
max,0.994945,0.995309
